# Bounds-theorem grid analysis

This notebook evaluates Taylor-reference and local-affine diagnostics from
`bounds_memo_v22.tex` across benchmark/transformation pairs that are compatible
with the current `sabench` registries.

The notebook is intentionally registry-driven. It relies on tested reusable
utilities in `sabench.analysis.bounds` and `sabench.analysis.bounds_grid`, not
notebook-local scientific logic.

Important interpretation note: default runs use empirical sample ranges as
support bounds. Those rows are labelled as sample-range diagnostics, not theorem
guarantees. The Taylor-bound comparison is against the Taylor reference `V_m`,
not directly against the original output `Y`.

## Status vocabulary

Rows use the following statuses: `bounds_supported`,
`bounds_diagnostic_sample_support`, `bounds_not_scalar_output`,
`bounds_not_pointwise`, `bounds_not_smooth`, `bounds_no_derivative_metadata`,
`bounds_domain_invalid`, `bounds_reference_zero_variance`, `bounds_eta_ge_one`,
and `bounds_failed`.

## Configuration

The default configuration evaluates a CI-safe smoke grid. Set
`SABENCH_BOUNDS_MAX_BENCHMARKS`, `SABENCH_BOUNDS_MAX_TRANSFORMS`,
`SABENCH_BOUNDS_N_BASE`, or `SABENCH_BOUNDS_OUTPUT_DIR` to adjust execution.

In [ ]:
from __future__ import annotations

import csv
import os
from collections import Counter
from pathlib import Path
from typing import Any

from sabench.analysis.bounds import supported_smooth_pointwise_transform_keys
from sabench.analysis.bounds_grid import BOUNDS_STATUSES, evaluate_bounds_grid
from sabench.benchmarks import BENCHMARK_REGISTRY
from sabench.transforms import TRANSFORM_REGISTRY

FAST_MODE = os.environ.get("SABENCH_BOUNDS_FAST_MODE", "1") != "0"
N_BASE = int(os.environ.get("SABENCH_BOUNDS_N_BASE", "128" if FAST_MODE else "4096"))
RNG_SEED = int(os.environ.get("SABENCH_BOUNDS_SEED", "20260429"))
TAYLOR_ORDER = int(os.environ.get("SABENCH_BOUNDS_TAYLOR_ORDER", "2"))
MAX_BENCHMARKS = int(os.environ.get("SABENCH_BOUNDS_MAX_BENCHMARKS", "4" if FAST_MODE else "0"))
MAX_TRANSFORMS = int(os.environ.get("SABENCH_BOUNDS_MAX_TRANSFORMS", "12" if FAST_MODE else "0"))
OUTPUT_DIR = Path(
    os.environ.get(
        "SABENCH_BOUNDS_OUTPUT_DIR",
        "outputs/notebooks/03_bounds_theorem_grid_analysis",
    )
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

benchmark_keys = tuple(BENCHMARK_REGISTRY)
supported_transform_keys = tuple(
    key for key in supported_smooth_pointwise_transform_keys() if key in TRANSFORM_REGISTRY
)
remaining_transform_keys = tuple(
    key for key in TRANSFORM_REGISTRY if key not in set(supported_transform_keys)
)
transform_keys = supported_transform_keys + remaining_transform_keys

if MAX_BENCHMARKS > 0:
    benchmark_keys = benchmark_keys[:MAX_BENCHMARKS]
if MAX_TRANSFORMS > 0:
    transform_keys = transform_keys[:MAX_TRANSFORMS]

print(
    f"Configured bounds grid with {len(benchmark_keys)} benchmarks, "
    f"{len(transform_keys)} transforms, N_BASE={N_BASE}, order={TAYLOR_ORDER}."
)

## Execute the theorem-assumption grid

Each result row represents one benchmark/transformation pair. Non-applicable
and failed pairs are retained as status rows rather than stopping the full grid.

In [ ]:
results = evaluate_bounds_grid(
    benchmark_keys=benchmark_keys,
    transform_keys=transform_keys,
    n_base=N_BASE,
    seed=RNG_SEED,
    taylor_order=TAYLOR_ORDER,
)
rows = [result.as_dict() for result in results]

status_counts = Counter(row["bounds_status"] for row in rows)
print(f"Evaluated {len(rows)} candidate pairs")
print(dict(sorted(status_counts.items())))

## Export diagnostic tables

`bounds_pair_status.csv` contains every candidate pair.
`taylor_reference_results.csv` contains rows with Taylor-reference diagnostics.
`local_affine_results.csv` contains rows with local-affine diagnostics.
`bounds_summary.csv` counts every supported status.

In [ ]:
PAIR_STATUS_COLUMNS = [
    "benchmark_key",
    "transform_key",
    "bounds_status",
    "reason",
    "benchmark_output_kind",
    "transform_mechanism",
    "transform_tags",
    "n_base",
    "seed",
    "n_inputs",
    "n_evaluations",
    "output_shape",
    "output_finite",
    "output_variance",
    "taylor_order",
]
TAYLOR_COLUMNS = [
    "benchmark_key",
    "transform_key",
    "bounds_status",
    "support_source",
    "support_lower",
    "support_upper",
    "taylor_status",
    "eta_empirical",
    "eta_sufficient",
    "eta_empirical_lt_one",
    "eta_sufficient_lt_one",
    "projection_bound_s1_max",
    "projection_bound_st_max",
    "projection_bound_s1_mean",
    "projection_bound_st_mean",
    "reference_shift_s1_max",
    "reference_shift_st_max",
    "reference_shift_s1_mean",
    "reference_shift_st_mean",
]
LOCAL_AFFINE_COLUMNS = [
    "benchmark_key",
    "transform_key",
    "bounds_status",
    "local_affine_status",
    "support_source",
    "sigma_y",
    "mu4",
    "phi_prime_mu",
    "rho2",
    "kappa",
    "moment_ratio",
    "lambda_value",
    "eta_upper",
    "lambda_lt_two",
]

taylor_rows = [row for row in rows if row.get("support_source")]
local_affine_rows = [row for row in rows if row.get("local_affine_status")]
summary_rows = [
    {"bounds_status": status, "count": status_counts.get(status, 0)}
    for status in BOUNDS_STATUSES
]

def write_csv(path: Path, table_rows: list[dict[str, Any]], columns: list[str]) -> None:
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(fieldnames=columns, f=handle, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(table_rows)

write_csv(OUTPUT_DIR / "bounds_pair_status.csv", rows, PAIR_STATUS_COLUMNS)
write_csv(OUTPUT_DIR / "taylor_reference_results.csv", taylor_rows, TAYLOR_COLUMNS)
write_csv(OUTPUT_DIR / "local_affine_results.csv", local_affine_rows, LOCAL_AFFINE_COLUMNS)
write_csv(OUTPUT_DIR / "bounds_summary.csv", summary_rows, ["bounds_status", "count"])

print(f"Wrote {OUTPUT_DIR / 'bounds_pair_status.csv'}")
print(f"Wrote {OUTPUT_DIR / 'taylor_reference_results.csv'}")
print(f"Wrote {OUTPUT_DIR / 'local_affine_results.csv'}")
print(f"Wrote {OUTPUT_DIR / 'bounds_summary.csv'}")

## Interpretation guardrails

- `bounds_supported` means caller-provided support bounds were used.
- `bounds_diagnostic_sample_support` means empirical sample-range diagnostics
  were computed and should not be reported as theorem guarantees.
- Projection-bound columns compare transformed-output profiles with Taylor
  reference `V_m` profiles.
- Local-affine columns report the nonzero-slope diagnostic quantities
  `lambda = K kappa`, `K = sqrt(mu4) / sigma_Y^2`, and
  `kappa = rho2 sigma_Y / |phi'(mu_Y)|`.